In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
# sys.path.append('/home/wolfgang/repos/sleep_research_io')
# from sleep_research_functions import *
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from icu_sleep_breathing_2020_help_functions import * 

sys.path.append('/home/wolfgang/repos/sleep_research_io/')
from sleep_research_functions import *
import re
from datetime import datetime, timedelta
import pdb
from sleep_analysis_functions import *

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300


font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 9}

font_subplots =  {'family' : 'sans-serif',
        'weight' : 'bold',
        'size'   : 9}

matplotlib.rc('font', **font)

In [2]:
plots_savedir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/plots'

plots_savedir = '/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Plots/swimmer'

In [3]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria()

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3338: DtypeWarning: Columns (15,91,92,93,6455,6456,6457,12819,12820,12821,19185,19186,19187,21491,21492,21493,21519,21520,21521,21526,21527,21528,25549,25550,25551,31913,31914,31915,38279,38280,38281,44643,44644,44645,46851,46852,46853,46865,46866,46867,46879,46880,46881,46949,46950,46951,46956,46957,46958,46963,46964,46965,46970,46971,46972,46977,46978,46979,46984,46985,46986,51007,51008,51009,53336,53337,53338,57305,57373,57374,57375,59588,59589,59590,59602,59603,59604,59686,59687,59688,63669,63737,63738,63739,65952,65953,65954,65959,65960,65961,65980,65981,65982,66036,66037,66038,66043,66044,66045,66050,66051,66052,66057,66058,66059,66071,66072,66073,66078,66079,66080,70033,70101,70102,70103,72318,72319,72320,72332,72333,72334,72402,72403,72404,72416,72417,72418,76399,76467,76468,76469,78682,78683,78684,78696,78697,78698,78710,78711,78712,78766,78767,78768,78787,78788,78789,78801,7880

# of subjects ICU before inclusion criteria: 102
# of 12-hour segments ICU before inclusion criteria: 1161
# of subjects ICU after inclusion criteria: 102
# of 12-hour segments ICU after inclusion criteria: 1161
24-hour segments: 387
12-hour segments: 774

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412


In [4]:
summary_days_icu.loc[:, ['stages_distribution' in x for x in summary_days_icu.columns]] *= 100
summary_days_sleeplab.loc[:, ['stages_distribution' in x for x in summary_days_sleeplab.columns]] *= 100

summary_f_icu = summary_days_icu.loc[summary_days_icu.day_cat == 'f', :]
summary_dn_icu = summary_days_icu.loc[summary_days_icu.day_cat != 'f', :]

summary_days_sleeplab_full = summary_days_sleeplab.copy()

In [5]:
#### select version of plot:
ahi_category = 'all'
agreement = 'agreement_relaxed'
icu_day_v = 'full'
####

summary_days_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_' + ahi_category] == 1]

min_hours_sleep_icu = 0.01
if icu_day_v == 'day_night':
    summary_dn_icu_sel = summary_dn_icu.loc[np.any([summary_dn_icu[f'hours_sleep_amendsumeffort_{agreement}'] >= min_hours_sleep_icu, 
                                                   summary_dn_icu[f'hours_sleep_ecg_nn_{agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :]
elif icu_day_v == 'full':
    summary_dn_icu_sel = summary_f_icu.loc[np.any([summary_f_icu[f'hours_sleep_amendsumeffort_{agreement}'] >= min_hours_sleep_icu, 
                                                   summary_f_icu[f'hours_sleep_ecg_nn_{agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :] 
    
summary_days_sleeplab = summary_days_sleeplab.loc[np.any([summary_days_sleeplab[f'hours_sleep_amendsumeffort_{agreement}'] >= min_hours_sleep_icu, 
                                                   summary_days_sleeplab[f'hours_sleep_ecg_nn_{agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :] 

In [6]:
icu_study_ids = summary_dn_icu_sel.study_id.unique()
icu_study_ids.sort()

In [7]:
dir_input = f'/scr/wolfgang/Sleep_And_Breathing/icu_files_step8'

files = os.listdir(dir_input)
files.sort()

In [8]:
print(len(files))
print(icu_study_ids.shape[0])

108
101


In [9]:
# we now use ALL included ICU patients, also those who do not show any sleep:
icu_study_ids = summary_subjects_icu.query("inclusion_subject == 1")['study_id']
print(icu_study_ids.shape[0])

102


In [10]:
cols_agree = ['ecg_nn_amendsumeffort_agreement', 'ecg_nn_amendsumeffort_agreement_relaxed']

stage_cols = ['stage_pred_amendsumeffort',
  'stage_pred_ecg_nn']

In [11]:
swimmer = pd.DataFrame(columns=pd.MultiIndex.from_product([[], []], names=['modality', 'study_id']))
alignment = True
agreement = 'ecg_nn_amendsumeffort_agreement_relaxed'

### sorting: read in files in correct order

In [12]:
sorting = 'discordance'
# read in the summary per subject, a result from icu_sleep_breathing_2020_2_sleep_stage_analysis_w_agreements
# mean statistic over all nights per subject.
summary_dn_icu_sel_subject = pd.read_csv('summary_icu_mean_per_subject.csv')
summary_dn_icu_sel_subject.sort_values(by = 'perc_discordant_mean_agreement_relaxed', ascending=True, inplace=True)
study_ids_sorted = list(summary_dn_icu_sel_subject.study_id.values)

icu_study_ids = np.array(study_ids_sorted).astype(int)
files = ['icusleep_' + str(study_id).zfill(3) +'.h5' for study_id in icu_study_ids]

In [13]:
len(summary_dn_icu_sel_subject.study_id.values)

102

In [14]:
summary_dn_icu_sel_subject.study_id.values

array([ 35.,  50.,  15.,  74., 123.,   1., 177.,   3.,  69.,  11., 165.,
       137.,  21.,  29.,  25., 128., 104., 149.,  52., 131., 121.,  68.,
        18., 157.,  51., 120.,  88., 171.,  13., 122.,  87., 183.,  14.,
        83.,  54., 129., 100.,  79.,  12., 164., 179.,  89.,  39.,  24.,
       124., 151.,  34.,  90., 109.,  48., 140.,  31.,  46.,  33., 180.,
        81., 168.,  49., 130., 173.,  47., 156., 145., 142.,   6.,  45.,
       167., 187., 160., 154., 143., 152.,  72., 111., 186., 189., 181.,
        77., 133.,  56., 178., 135.,  91., 161., 139., 188., 158., 175.,
       182.,  37., 153., 155., 126.,  84., 172., 103., 176.,  82.,  63.,
        86., 101., 169.])

In [15]:
if 0:
    
    for file in files:

    #     try:
        filepath = os.path.join(dir_input, file)
        study_id = re.search('\d\d\d', file)[0]
        if not int(study_id) in list(icu_study_ids):
            continue

        data, hdr = load_sleep_data(filepath, load_all_signals = 0, idx_to_datetime = 1, 
        signals_to_load = stage_cols + cols_agree, load_events = 0, verbose = False)

        data = data.loc[data['stage_pred_amendsumeffort'].dropna().index[0].date():] # start with airgo first day available.
        data = data.loc[data.index[0].ceil('1min'):, :] # start with round minute in all patients
        data = data.iloc[::30 * hdr['fs'], :] # 30-second-epochs
        data = data.dropna(how='all') # remove data where there is no sleep stage at all available

        data_agrelaxed = data.copy()
        data = data[stage_cols] # data ignoring agreement
        
        # NAN replacement for data where only one modality is available.
        data.loc[~np.isin(data_agrelaxed[agreement], [0, 1]), :] = np.nan # NAN where no agreement (so only overlap data is considered)

        data_agrelaxed.loc[~np.isin(data_agrelaxed[agreement], [0, 1]), :] = np.nan # NAN where no agreement (so only overlap data is considered)
        data_agrelaxed.loc[data_agrelaxed[agreement]==0, :] = 6 # 6 where disagreement
        data_agrelaxed.drop(cols_agree, axis=1, inplace=True)
        data_agrelaxed.columns = [x + '_agrelaxed' for x in data_agrelaxed.columns]
        data = data.join(data_agrelaxed)

        if alignment:
            days_subtract_align = (data.index[0] - pd.Timestamp('2020-01-01')).days
            data.index = data.index - timedelta(days=days_subtract_align)

        data['zzznan'] = np.nan

        data.columns = pd.MultiIndex.from_product([data.columns, [study_id]], names=['modality', 'study_id'])
        swimmer = swimmer.join(data, how='outer')

    swimmer = swimmer.loc[:'2020-01-08']
    swimmer = swimmer.fillna(-1).transpose()

    pickle.dump(swimmer , open(os.path.join(plots_savedir, f"swimmer_icu_sleep_breathing_2020_sorted{sorting}.p"), "wb" ) )

else:
    
    with open(os.path.join(plots_savedir,f"swimmer_icu_sleep_breathing_2020_sorted{sorting}.p"), 'rb') as handle:
        swimmer = pickle.load(handle)
    

In [16]:
# data, hdr = load_sleep_data(filepath, load_all_signals = 0, idx_to_datetime = 1, 
# signals_to_load = stage_cols + cols_agree, load_events = 0, verbose = False)

In [17]:
# data.ecg_nn_amendsumeffort_agreement.dropna()

In [18]:
# data.stage_pred_amendsumeffort_agrelaxed.dropna()

In [19]:
# swimmer.index.get_level_values('study_id').unique()

In [20]:
# swimmer

In [21]:
# # for 126, see what happens.

# study_id_tmp = 122

In [22]:
# filepath = f'/scr/wolfgang/Sleep_And_Breathing/icu_files_step8/icusleep_{study_id_tmp}.h5'
# data, hdr = load_sleep_data(filepath, load_all_signals = 0, idx_to_datetime = 1, 
#                             signals_to_load = stage_cols + cols_agree, load_events = 0, verbose = False)

# data = data.loc[data['stage_pred_amendsumeffort'].dropna().index[0].date():] # start with airgo first day available.
# data = data.loc[data.index[0].ceil('1min'):, :] # start with round minute in all patients
# data = data.iloc[::30 * hdr['fs'], :] # 30-second-epochs
# data = data.dropna(how='all') # remove data where there is no sleep stage at all available


In [23]:
# data.loc[data.stage_pred_amendsumeffort.notna() & data.stage_pred_ecg_nn.notna()]

In [24]:
# [x for x in summary_days_icu.columns if 'hours_sleep_amendsumeffort' in x]

In [25]:
# inclusion_agreement='all'
# print(summary_days_icu[summary_days_icu.study_id == study_id_tmp][f'hours_sleep_amendsumeffort_{inclusion_agreement}'].values)
# print(summary_days_icu[summary_days_icu.study_id == study_id_tmp][f'hours_sleep_ecg_nn_{inclusion_agreement}'].values)

In [26]:
# inclusion_agreement='agreement_relaxed'
# print(summary_days_icu[summary_days_icu.study_id == study_id_tmp][f'hours_sleep_amendsumeffort_{inclusion_agreement}'].values)
# print(summary_days_icu[summary_days_icu.study_id == study_id_tmp][f'hours_sleep_ecg_nn_{inclusion_agreement}'].values)

In [27]:
# booli = pd.notna(data['stage_pred_amendsumeffort']).values.flatten()

In [28]:
# (data[booli]['stage_pred_amendsumeffort'] < 5).sum() / 2 / 60

In [29]:
# (data[booli]['stage_pred_ecg_nn'] < 5).sum() / 2 / 60

In [30]:
# ((data[booli]['stage_pred_ecg_nn'] < 5) & (data[booli]['stage_pred_amendsumeffort'] < 5)).sum() / 2 / 60

In [31]:
# [x for x in data.columns if 'stage' in x]

In [32]:
# np.unique(data_agrelaxed.values.flatten())[:7]

In [33]:
# swimmer.head()

In [34]:
# swimmer[('stage_pred_amendsumeffort', '001')].dropna()

In [185]:
def colors_colorblind():
    
    black = (0, 0, 0)
    orange = (0.9, 0.6, 0)
    skyblue = (0.35, 0.75, 0.95)
    bluegreen = (0, 0.67, 0.45)
    yellow = (0.95, 0.9, 0.15)
    blue = (0, 0.45, 0.7)
    red = (0.9, 0.2, 0)
    purple = (0.85, 0.65, 0.75)
    
    return [black, orange, skyblue, bluegreen, yellow, blue, red, purple]

In [188]:
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

colors = ['white', 'white', 'blue', 'black', 'lightblue', (0/255, 158/255, 115/255), (227/255, 66/255, 52/255), 'orange']
colors = ['white', 'white', 'blue', 'blue', 'lightblue', 'limegreen', 'red', 'orange']

[black, orange, skyblue, bluegreen, yellow, blue, red, purple] = colors_colorblind()
colors = ['white', 'white', blue, 'k', bluegreen, skyblue, red, orange]

# colors = [black, orange, skyblue, bluegreen, yellow, blue, red, purple]
colors = ['white', 'white',  black, blue, skyblue, bluegreen, red, orange]

cm = LinearSegmentedColormap.from_list("cm_stages", colors, N=len(colors))

# cm_stages_discordance = ['white', 'blue', 'black', 'deepskyblue', 'limegreen', 'red', 'orange']
# cm_stages_discordance = LinearSegmentedColormap.from_list("cm_greens", cm_stages_discordance)


In [189]:
test = np.expand_dims(np.arange(-1, 7), 0)

fig, ax = plt.subplots(1, 1, figsize=(5, 1));
ax.matshow(test, cmap=cm)
ax.set_xticks(np.arange(len(test.flatten())))
ax.set_xticklabels(test.flatten());

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [217]:
swimmer_3_stages = swimmer.copy()
swimmer_3_stages[swimmer_3_stages == 1] = 2
swimmer_3_stages[swimmer_3_stages == 3] = 2
# swimmer_3_stages[swimmer_3_stages == 4] = 2
# swimmer_3_stages[swimmer_3_stages == 5] = 3


In [218]:
np.unique(swimmer_3_stages.values)

array([-1.,  2.,  4.,  5.,  6.])

In [219]:
swimmer_3_stages;

In [220]:
swimmer.transpose().index[0]

Timestamp('2020-01-01 00:00:00')

In [221]:
idx_day0_8am = np.where(swimmer.transpose().index > '2020-01-01 08:00:00')[0][0]
idx_day0_8pm = np.where(swimmer.transpose().index > '2020-01-01 20:00:00')[0][0]
idx_day0_midnight = np.where(swimmer.transpose().index > '2020-01-01 00:00:00')[0][0]

# restrict swimmer to number of days:
idx_day_max = np.where(swimmer.transpose().index > '2020-01-06 00:01:00')[0][0]

In [254]:
from matplotlib.lines import Line2D

# version = 'NREM_R_W'
version = '5_stages'
stage_columns = ['stage_pred_amendsumeffort', 'stage_pred_ecg_nn']
# stage_columns = ['stage_pred_amendsumeffort']
# stage_columns = ['stage_pred_ecg_nn']
# stage_columns = ['stage_pred_ecg_nn_agrelaxed']
# stage_columns = ['stage_pred_amendsumeffort_agrelaxed']
# stage_columns = ['stage_pred_amendsumeffort_agrelaxed', 'stage_pred_ecg_nn_agrelaxed']

legend_lines = []
if version == 'NREM_R_W':
    swimmer_plot = swimmer_3_stages
    legend_lines.append(Line2D([0], [0], color=colors[3], lw=4))
    legend_lines.append(Line2D([0], [0], color=colors[5], lw=4)) 
    legend_lines.append(Line2D([0], [0], color=colors[6], lw=4))
    legend_names = ['NREM', 'REM', 'Wake']

elif version == '5_stages':
    swimmer_plot = swimmer
    for i in range(2, 7):
        legend_lines.append(Line2D([0], [0], color=colors[i], lw=4))
        legend_names = ['N3', 'N2', 'N1', 'REM', 'Wake']
    
if len(stage_columns) == 1:
    figsize_y = 9 # 8 
if len(stage_columns) == 2:
    figsize_y = 9 # 15
    
savename = 'Figure_Swimmer_'
if stage_columns == ['stage_pred_amendsumeffort']:
    title = 'Sleep Stages based on Breathing'
    savename += 'breathing_' + version
#     vmax = 5
elif stage_columns == ['stage_pred_amendsumeffort_agrelaxed']:
    title = 'Sleep Stages based on Breathing'
    savename += 'breathing_agrelaxed_' + version
#     vmax = 6
#     cm = cm_stages_discordance
    legend_lines.append(Line2D([0], [0], color='orange', lw=4))
    legend_names.append('Discordant')
elif stage_columns == ['stage_pred_ecg_nn']:
    title = 'Sleep Stages in the ICU based on Heart Rate Variability'
    savename += 'hrv_' + version
#     vmax = 5
#     cm = cm_stages
elif stage_columns == ['stage_pred_ecg_nn_agrelaxed']:
    title = 'Sleep Stages in the ICU based on Heart Rate Variability'
    savename += 'hrv_agrelaxed' + version
#     vmax = 6
#     cm = cm_stages_discordance
    legend_lines.append(Line2D([0], [0], color='orange', lw=4))
    legend_names.append('Discordant')

elif stage_columns == ['stage_pred_amendsumeffort', 'stage_pred_ecg_nn']:
    title = 'Sleep Stages in the ICU based on Breathing and Heart Rate Variability'
    savename += 'breathing_hrv_' + version
#     vmax = 5
#     cm = cm_stages
elif stage_columns == ['stage_pred_amendsumeffort_agrelaxed', 'stage_pred_ecg_nn_agrelaxed']:
    title = 'Sleep Stages in the ICU based on Breathing and Heart Rate Variability'
    savename += 'breathing_hrv_agrelaxed' + version
#     vmax = 6
#     cm = cm_stages_discordance
    legend_lines.append(Line2D([0], [0], color='orange', lw=4))
    legend_names.append('Discordant')
    
else:
    raise ValueError('Wrong stage_columns')

    
swimmer_plot = swimmer_plot.iloc[:, :idx_day_max]
    
legend_lines += [plt.vlines(0, 0, 1, colors='k', alpha=0.75, linestyles='dashed', lw=0.75)];
legend_names += ['Midnight']

In [255]:
swimmer_plot = swimmer_plot.reset_index()
swimmer_plot = swimmer_plot[np.isin(swimmer_plot.modality, stage_columns)].set_index(['modality', 'study_id'])

In [256]:
# landscape: 8.5 x 7
fig,ax = plt.subplots(figsize=(7.5, figsize_y))

ax.matshow(swimmer_plot,  # swimmer_plot.loc[stage_columns]
           cmap=cm, vmin=-1, vmax=6, alpha=1,
           aspect='auto')

x_tick_interval_hours = 24
x_8am = np.arange(idx_day0_8am, swimmer_plot.shape[1] - idx_day0_8am, 2*60*x_tick_interval_hours)
x_8pm = np.arange(idx_day0_8pm, swimmer_plot.shape[1] - idx_day0_8pm - 1, 2*60*x_tick_interval_hours)
x_midnight = np.arange(idx_day0_midnight, swimmer_plot.shape[1] - idx_day0_midnight, 2*60*x_tick_interval_hours)

x_ticks_major = x_midnight
ax.set_xticks(x_ticks_major)
ax.set_xticklabels(['Day ' + str(x) for x in np.arange(len(x_ticks_major) - 1) + 1] + [''],
                  fontsize=8, ha='left')

# ax.set_xticks(x_midnight, minor=True)
# ax.set_xticklabels(['12am'] * len(x_midnight), minor=True, fontsize=8)

# ax.set_xticklabels(['12am'] * np.arange(x_midnight, swimmer.shape[1] - x_midnight - 1, 2*60*x_tick_interval_hours).shape[0], minor=True)
ax.vlines(x_midnight[:-1], swimmer_plot.shape[0], 0, 
          colors='k', alpha=0.75, linestyles='dashed', lw=0.75)


ax.tick_params(axis="x", bottom=True, top=False, labelbottom=True, labeltop=False, which='both', length=2)
ax.tick_params(axis="y", length=2)

if len(stage_columns) == 2:
    ax.set_yticks(0.5 + 2 * np.arange(len(icu_study_ids)))
    yticklabels_all_patients = np.arange(len(icu_study_ids)) + 1
    yticklabels = [x if x%5==0 else '' for x in yticklabels_all_patients]
    yticklabels[0] = 1
    ax.set_yticklabels(yticklabels)
    
# ax.set_xlim([0, x_ticks_major[-1]])
ax.set_ylabel('Patient #')
ax.set_xlabel('Time', labelpad=1)
# ax.set_title(title, pad=-20)
ax.legend(legend_lines, legend_names, ncol=len(legend_lines) , bbox_to_anchor=(1, 1.05), bbox_transform=ax.transAxes, frameon=False)
plt.tight_layout()


ax.set_ylim([swimmer_plot.shape[0], -0.6])

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.savefig(os.path.join(plots_savedir, savename + '.tiff'), dpi=600)
plt.savefig(os.path.join(plots_savedir, savename + '.svg'))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [257]:
plots_savedir

'/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Plots/swimmer'

In [41]:
savename

'Figure_Swimmer_breathing_hrv_5_stages'

In [42]:
plt.close('all')

In [43]:
stage_columns = ['stage_pred_ecg_nn', 'stage_pred_amendsumeffort']

In [44]:
swimmer = swimmer.drop('zzznan', axis=0)

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/pandas/core/generic.py:4150: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  obj = obj._drop_axis(labels, axis, level=level, errors=errors)


In [45]:
swimmer.loc[stage_columns].columns

DatetimeIndex(['2020-01-01 00:00:00', '2020-01-01 00:00:30',
               '2020-01-01 00:01:00', '2020-01-01 00:01:30',
               '2020-01-01 00:02:00', '2020-01-01 00:02:30',
               '2020-01-01 00:03:00', '2020-01-01 00:03:30',
               '2020-01-01 00:04:00', '2020-01-01 00:04:30',
               ...
               '2020-01-08 23:55:00', '2020-01-08 23:55:30',
               '2020-01-08 23:56:00', '2020-01-08 23:56:30',
               '2020-01-08 23:57:00', '2020-01-08 23:57:30',
               '2020-01-08 23:58:00', '2020-01-08 23:58:30',
               '2020-01-08 23:59:00', '2020-01-08 23:59:30'],
              dtype='datetime64[ns]', length=23040, freq=None)

In [46]:
swimmer.max().max()

6.0

In [47]:
swimmer.loc[stage_columns].max().max()

5.0

In [48]:
day_cols = [(x.hour >= 8) & (x.hour < 20) for x in swimmer.loc[stage_columns].columns]
night_cols = [(x.hour < 8) | (x.hour >= 20) for x in swimmer.loc[stage_columns].columns]

# swimmer_plot.loc[stage_columns, day_cols]

In [49]:
swimmer

,,2020-01-01 00:00:00,2020-01-01 00:00:30,2020-01-01 00:01:00,2020-01-01 00:01:30,2020-01-01 00:02:00,2020-01-01 00:02:30,2020-01-01 00:03:00,2020-01-01 00:03:30,2020-01-01 00:04:00,2020-01-01 00:04:30,2020-01-01 00:05:00,2020-01-01 00:05:30,2020-01-01 00:06:00,2020-01-01 00:06:30,2020-01-01 00:07:00,2020-01-01 00:07:30,2020-01-01 00:08:00,2020-01-01 00:08:30,2020-01-01 00:09:00,2020-01-01 00:09:30,2020-01-01 00:10:00,2020-01-01 00:10:30,2020-01-01 00:11:00,2020-01-01 00:11:30,2020-01-01 00:12:00,2020-01-01 00:12:30,2020-01-01 00:13:00,2020-01-01 00:13:30,2020-01-01 00:14:00,2020-01-01 00:14:30,2020-01-01 00:15:00,2020-01-01 00:15:30,2020-01-01 00:16:00,2020-01-01 00:16:30,2020-01-01 00:17:00,2020-01-01 00:17:30,2020-01-01 00:18:00,2020-01-01 00:18:30,2020-01-01 00:19:00,2020-01-01 00:19:30,2020-01-01 00:20:00,2020-01-01 00:20:30,2020-01-01 00:21:00,2020-01-01 00:21:30,2020-01-01 00:22:00,2020-01-01 00:22:30,2020-01-01 00:23:00,2020-01-01 00:23:30,2020-01-01 00:24:00,2020-01-01 00:24:30,2020-01-01 00:25:00,2020-01-01 00:25:30,2020-01-01 00:26:00,2020-01-01 00:26:30,2020-01-01 00:27:00,2020-01-01 00:27:30,2020-01-01 00:28:00,2020-01-01 00:28:30,2020-01-01 00:29:00,2020-01-01 00:29:30,2020-01-01 00:30:00,2020-01-01 00:30:30,2020-01-01 00:31:00,2020-01-01 00:31:30,2020-01-01 00:32:00,2020-01-01 00:32:30,2020-01-01 00:33:00,2020-01-01 00:33:30,2020-01-01 00:34:00,2020-01-01 00:34:30,2020-01-01 00:35:00,2020-01-01 00:35:30,2020-01-01 00:36:00,2020-01-01 00:36:30,2020-01-01 00:37:00,2020-01-01 00:37:30,2020-01-01 00:38:00,2020-01-01 00:38:30,2020-01-01 00:39:00,2020-01-01 00:39:30,2020-01-01 00:40:00,2020-01-01 00:40:30,2020-01-01 00:41:00,2020-01-01 00:41:30,2020-01-01 00:42:00,2020-01-01 00:42:30,2020-01-01 00:43:00,2020-01-01 00:43:30,2020-01-01 00:44:00,2020-01-01 00:44:30,2020-01-01 00:45:00,2020-01-01 00:45:30,2020-01-01 00:46:00,2020-01-01 00:46:30,2020-01-01 00:47:00,2020-01-01 00:47:30,2020-01-01 00:48:00,2020-01-01 00:48:30,2020-01-01 00:49:00,2020-01-01 00:49:30,2020-01-01 00:50:00,2020-01-01 00:50:30,2020-01-01 00:51:00,2020-01-01 00:51:30,2020-01-01 00:52:00,2020-01-01 00:52:30,2020-01-01 00:53:00,2020-01-01 00:53:30,2020-01-01 00:54:00,2020-01-01 00:54:30,2020-01-01 00:55:00,2020-01-01 00:55:30,2020-01-01 00:56:00,2020-01-01 00:56:30,2020-01-01 00:57:00,2020-01-01 00:57:30,2020-01-01 00:58:00,2020-01-01 00:58:30,2020-01-01 00:59:00,2020-01-01 00:59:30,2020-01-01 01:00:00,2020-01-01 01:00:30,2020-01-01 01:01:00,2020-01-01 01:01:30,2020-01-01 01:02:00,2020-01-01 01:02:30,2020-01-01 01:03:00,2020-01-01 01:03:30,2020-01-01 01:04:00,2020-01-01 01:04:30,2020-01-01 01:05:00,2020-01-01 01:05:30,2020-01-01 01:06:00,2020-01-01 01:06:30,2020-01-01 01:07:00,2020-01-01 01:07:30,2020-01-01 01:08:00,2020-01-01 01:08:30,2020-01-01 01:09:00,2020-01-01 01:09:30,2020-01-01 01:10:00,2020-01-01 01:10:30,2020-01-01 01:11:00,2020-01-01 01:11:30,2020-01-01 01:12:00,2020-01-01 01:12:30,2020-01-01 01:13:00,2020-01-01 01:13:30,2020-01-01 01:14:00,2020-01-01 01:14:30,...,2020-01-08 22:45:00,2020-01-08 22:45:30,2020-01-08 22:46:00,2020-01-08 22:46:30,2020-01-08 22:47:00,2020-01-08 22:47:30,2020-01-08 22:48:00,2020-01-08 22:48:30,2020-01-08 22:49:00,2020-01-08 22:49:30,2020-01-08 22:50:00,2020-01-08 22:50:30,2020-01-08 22:51:00,2020-01-08 22:51:30,2020-01-08 22:52:00,2020-01-08 22:52:30,2020-01-08 22:53:00,2020-01-08 22:53:30,2020-01-08 22:54:00,2020-01-08 22:54:30,2020-01-08 22:55:00,2020-01-08 22:55:30,2020-01-08 22:56:00,2020-01-08 22:56:30,2020-01-08 22:57:00,2020-01-08 22:57:30,2020-01-08 22:58:00,2020-01-08 22:58:30,2020-01-08 22:59:00,2020-01-08 22:59:30,2020-01-08 23:00:00,2020-01-08 23:00:30,2020-01-08 23:01:00,2020-01-08 23:01:30,2020-01-08 23:02:00,2020-01-08 23:02:30,2020-01-08 23:03:00,2020-01-08 23:03:30,2020-01-08 23:04:00,2020-01-08 23:04:30,2020-01-08 23:05:00,2020-01-08 23:05:30,2020-01-08 23:06:00,2020-01-08 23:06:30,2020-01-08 23:07:00,2020-01-08 23:07:30,2020-01-08 23:08:00,2020-01-08 23:08:30,2020-01-08 23:09:00,2020-01-08 23:

In [50]:
swimmer_D = pd.DataFrame(data=np.isin(swimmer, [1, 2, 3, 4, 5]).astype(int), columns=swimmer.columns, index=swimmer.index).transpose().resample('H').sum() # data vailable
swimmer_S = pd.DataFrame(data=np.isin(swimmer, [1, 2, 3, 4]).astype(int), columns=swimmer.columns, index=swimmer.index).transpose().resample('H').sum()
swimmer_W = pd.DataFrame(data=np.isin(swimmer, [5]).astype(int), columns=swimmer.columns, index=swimmer.index).transpose().resample('H').sum()
swimmer_R = pd.DataFrame(data=np.isin(swimmer, [4]).astype(int), columns=swimmer.columns, index=swimmer.index).transpose().resample('H').sum()

swimmer_D_night = pd.DataFrame(data=np.isin(swimmer.loc[:, night_cols], [1, 2, 3, 4, 5]).astype(int), columns=swimmer.loc[:, night_cols].columns, index=swimmer.index).transpose().resample('H').sum()
swimmer_S_night = pd.DataFrame(data=np.isin(swimmer.loc[:, night_cols], [1, 2, 3, 4]).astype(int), columns=swimmer.loc[:, night_cols].columns, index=swimmer.index).transpose().resample('H').sum()
swimmer_W_night = pd.DataFrame(data=np.isin(swimmer.loc[:, night_cols], [5]).astype(int), columns=swimmer.loc[:, night_cols].columns, index=swimmer.index).transpose().resample('H').sum()
swimmer_R_night = pd.DataFrame(data=np.isin(swimmer.loc[:, night_cols], [4]).astype(int), columns=swimmer.loc[:, night_cols].columns, index=swimmer.index).transpose().resample('H').sum()

swimmer_D_day = pd.DataFrame(data=np.isin(swimmer.loc[:, day_cols], [1, 2, 3, 4, 5]).astype(int), columns=swimmer.loc[:, day_cols].columns, index=swimmer.index).transpose().resample('H').sum()
swimmer_S_day = pd.DataFrame(data=np.isin(swimmer.loc[:, day_cols], [1, 2, 3, 4]).astype(int), columns=swimmer.loc[:, day_cols].columns, index=swimmer.index).transpose().resample('H').sum()
swimmer_W_day = pd.DataFrame(data=np.isin(swimmer.loc[:, day_cols], [5]).astype(int), columns=swimmer.loc[:, day_cols].columns, index=swimmer.index).transpose().resample('H').sum()
swimmer_R_day = pd.DataFrame(data=np.isin(swimmer.loc[:, day_cols], [4]).astype(int), columns=swimmer.loc[:, day_cols].columns, index=swimmer.index).transpose().resample('H').sum()

sw_hours = pd.DataFrame(index = swimmer_S.sum().index)
sw_hours['data'] = swimmer_D.sum().values
sw_hours['sleep'] = swimmer_S.sum().values
sw_hours['wake'] = swimmer_W.sum().values
sw_hours['R'] = swimmer_R.sum().values

sw_hours_daynight = pd.DataFrame(index = swimmer_S.sum().index)
sw_hours['night_data'] = swimmer_D_night.sum().values
sw_hours['night_sleep'] = swimmer_S_night.sum().values
sw_hours['night_wake'] = swimmer_W_night.sum().values

sw_hours['day_data'] = swimmer_D_day.sum().values
sw_hours['day_sleep'] = swimmer_S_day.sum().values
sw_hours['day_wake'] = swimmer_W_day.sum().values

sw_hours['day_r'] = swimmer_R_day.sum().values
sw_hours['night_r'] = swimmer_R_night.sum().values

sw_hours['day_sleep_proportion'] = (sw_hours['day_sleep'].values / (sw_hours['day_wake'].values + sw_hours['day_sleep'].values) * 100).round(2)
sw_hours['night_sleep_proportion'] = (sw_hours['night_sleep'].values / (sw_hours['night_wake'].values + sw_hours['night_sleep'].values) * 100).round(2)

sw_hours['sleep_day_proportion'] = (sw_hours['day_sleep'].values / (sw_hours['day_sleep'].values + sw_hours['night_sleep'].values) * 100).round(2)
sw_hours['sleep_night_proportion'] = (sw_hours['night_sleep'].values / (sw_hours['day_sleep'].values + sw_hours['night_sleep'].values) * 100).round(2)

sw_hours['day_sleep_per24hours'] = sw_hours['day_sleep'].values / (sw_hours['day_data'].values * 2 * 60 / 24)
sw_hours['day_wake_per24hours'] = sw_hours['day_wake'].values / (sw_hours['day_data'].values * 2 * 60 / 24)
sw_hours['night_sleep_per24hours'] = sw_hours['night_sleep'].values / (sw_hours['night_data'].values * 2 * 60 / 24)
sw_hours['night_wake_per24hours'] = sw_hours['night_wake'].values / (sw_hours['night_data'].values * 2 * 60 / 24)
sw_hours['day_r_per24hours'] = sw_hours['day_r'].values / (sw_hours['day_data'].values * 2 * 60 / 24)
sw_hours['night_r_per24hours'] = sw_hours['night_r'].values / (sw_hours['night_data'].values * 2 * 60 / 24)

sw_hours['sleep_day_proportion_adjusted24hours'] = (sw_hours['day_sleep_per24hours'].values / (sw_hours['day_sleep_per24hours'].values + sw_hours['night_sleep_per24hours'].values) * 100).round(2)
sw_hours['sleep_night_proportion_adjusted24hours'] = (sw_hours['night_sleep_per24hours'].values / (sw_hours['day_sleep_per24hours'].values + sw_hours['night_sleep_per24hours'].values) * 100).round(2)

sw_hours_rem_occuring = sw_hours[sw_hours.R > 2].copy()
sw_hours_rem_occuring['rem_day_proportion'] = (sw_hours_rem_occuring['day_r'].values / (sw_hours_rem_occuring['day_r'].values + sw_hours_rem_occuring['night_r'].values) * 100).round(2)
sw_hours_rem_occuring['rem_night_proportion'] = (sw_hours_rem_occuring['night_r'].values / (sw_hours_rem_occuring['day_r'].values + sw_hours_rem_occuring['night_r'].values) * 100).round(2)

sw_hours_rem_occuring['rem_day_proportion_adjusted24hours'] = (sw_hours_rem_occuring['day_r_per24hours'].values / (sw_hours_rem_occuring['day_r_per24hours'].values + sw_hours_rem_occuring['night_r_per24hours'].values) * 100).round(2)
sw_hours_rem_occuring['rem_night_proportion_adjusted24hours'] = (sw_hours_rem_occuring['night_r_per24hours'].values / (sw_hours_rem_occuring['day_r_per24hours'].values + sw_hours_rem_occuring['night_r_per24hours'].values) * 100).round(2)

<ipython-input-50-d1ed6a51a762>:34: RuntimeWarning: invalid value encountered in true_divide
  sw_hours['day_sleep_proportion'] = (sw_hours['day_sleep'].values / (sw_hours['day_wake'].values + sw_hours['day_sleep'].values) * 100).round(2)
<ipython-input-50-d1ed6a51a762>:35: RuntimeWarning: invalid value encountered in true_divide
  sw_hours['night_sleep_proportion'] = (sw_hours['night_sleep'].values / (sw_hours['night_wake'].values + sw_hours['night_sleep'].values) * 100).round(2)
<ipython-input-50-d1ed6a51a762>:37: RuntimeWarning: invalid value encountered in true_divide
  sw_hours['sleep_day_proportion'] = (sw_hours['day_sleep'].values / (sw_hours['day_sleep'].values + sw_hours['night_sleep'].values) * 100).round(2)
<ipython-input-50-d1ed6a51a762>:38: RuntimeWarning: invalid value encountered in true_divide
  sw_hours['sleep_night_proportion'] = (sw_hours['night_sleep'].values / (sw_hours['day_sleep'].values + sw_hours['night_sleep'].values) * 100).round(2)
<ipython-input-50-d1ed6a51

In [51]:
# no sleep patients

In [52]:
print(sw_hours.shape)
print(sw_hours_rem_occuring.shape)

(408, 24)
(276, 28)


In [53]:
sw_hours.loc['stage_pred_amendsumeffort_agrelaxed'].head()

,data,sleep,wake,R,night_data,night_sleep,night_wake,day_data,day_sleep,day_wake,day_r,night_r,day_sleep_proportion,night_sleep_proportion,sleep_day_proportion,sleep_night_proportion,day_sleep_per24hours,day_wake_per24hours,night_sleep_per24hours,night_wake_per24hours,day_r_per24hours,night_r_per24hours,sleep_day_proportion_adjusted24hours,sleep_night_proportion_adjusted24hours
study_id,,,,,,,,,,,,,,,,,,,,,,,,
035,2038,978,1060,0,1391,888,503,647,90,557,0,0,13.91,63.84,9.20,90.80,0.027821,0.172179,0.127678,0.072322,0.000000,0.000000,17.89,82.11
050,1341,689,652,0,1087,683,404,254,6,248,0,0,2.36,62.83,0.87,99.13,0.004724,0.195276,0.125667,0.074333,0.000000,0.000000,3.62,96.38
001,1068,514,554,15,682,408,274,386,106,280,0,15,27.46,59.82,20.62,79.38,0.054922,0.145078,0.119648,0.080352,0.000000,0.004399,31.46,68.54
015,10749,2599,8150,11,5295,2254,3041,5454,345,5109,2,9,6.33,42.57,13.27,86.73,0.012651,0.187349,0.085137,0.114863,0.000073,0.000340,12.94,87.06
123,1468,1015,453,0,839,812,27,629,203,426,0,0,32.27,96.78,20.00,80.00,0.064547,0.135453,0.193564,0.006436,0.000000,0.000000,25.01,74.99


In [54]:

for stage_col in ['stage_pred_amendsumeffort', 'stage_pred_amendsumeffort_agrelaxed',
       'stage_pred_ecg_nn', 'stage_pred_ecg_nn_agrelaxed']:
    print(f"\n{stage_col}")
    print(f"Day sleep proportion:                     {sw_hours.loc[stage_col, 'day_sleep_proportion'].mean().round(2)} ({sw_hours.loc[stage_col, 'day_sleep_proportion'].std().round(2)})")
    print(f"Night sleep proportion:                   {sw_hours.loc[stage_col, 'night_sleep_proportion'].mean().round(2)} ({sw_hours.loc[stage_col, 'night_sleep_proportion'].std().round(2)})")
    print(f"Sleep day proportion:                     {sw_hours.loc[stage_col, 'sleep_day_proportion'].mean().round(2)} ({sw_hours.loc[stage_col, 'sleep_day_proportion'].std().round(2)})")
    print(f"Sleep night proportion:                   {sw_hours.loc[stage_col, 'sleep_night_proportion'].mean().round(2)} ({sw_hours.loc[stage_col, 'sleep_night_proportion'].std().round(2)})")
    print(f"Sleep day proportion - 24hour adjusted:   {sw_hours.loc[stage_col, 'sleep_day_proportion_adjusted24hours'].mean().round(2)} ({sw_hours.loc[stage_col, 'sleep_day_proportion_adjusted24hours'].std().round(2)})")
    print(f"Sleep night proportion - 24hour adjusted: {sw_hours.loc[stage_col, 'sleep_night_proportion_adjusted24hours'].mean().round(2)} ({sw_hours.loc[stage_col, 'sleep_night_proportion_adjusted24hours'].std().round(2)})")
    print(f"REM day proportion:                       {sw_hours_rem_occuring.loc[stage_col, 'rem_day_proportion'].mean().round(2)} ({sw_hours_rem_occuring.loc[stage_col, 'rem_day_proportion'].std().round(2)})")
    print(f"REM night proportion:                     {sw_hours_rem_occuring.loc[stage_col, 'rem_night_proportion'].mean().round(2)} ({sw_hours_rem_occuring.loc[stage_col, 'rem_night_proportion'].std().round(2)})")
    print(f"REM day proportion - 24hour adjusted:     {sw_hours_rem_occuring.loc[stage_col, 'rem_day_proportion_adjusted24hours'].mean().round(2)} ({sw_hours_rem_occuring.loc[stage_col, 'rem_day_proportion_adjusted24hours'].std().round(2)})")
    print(f"REM night proportion - 24hour adjusted:   {sw_hours_rem_occuring.loc[stage_col, 'rem_night_proportion_adjusted24hours'].mean().round(2)} ({sw_hours_rem_occuring.loc[stage_col, 'rem_night_proportion_adjusted24hours'].std().round(2)})")
        
    
    


stage_pred_amendsumeffort
Day sleep proportion:                     56.45 (37.06)
Night sleep proportion:                   73.11 (24.91)
Sleep day proportion:                     30.93 (17.04)
Sleep night proportion:                   69.07 (17.04)
Sleep day proportion - 24hour adjusted:   38.55 (16.43)
Sleep night proportion - 24hour adjusted: 61.45 (16.43)
REM day proportion:                       35.97 (28.54)
REM night proportion:                     64.03 (28.54)
REM day proportion - 24hour adjusted:     42.16 (28.43)
REM night proportion - 24hour adjusted:   57.84 (28.43)

stage_pred_amendsumeffort_agrelaxed
Day sleep proportion:                     45.84 (35.25)
Night sleep proportion:                   67.44 (27.53)
Sleep day proportion:                     28.95 (19.65)
Sleep night proportion:                   71.05 (19.65)
Sleep day proportion - 24hour adjusted:   35.22 (19.33)
Sleep night proportion - 24hour adjusted: 64.78 (19.33)
REM day proportion:                     

In [55]:
sw_hours.to_csv('icu_sleep_breathing_sw_hours.csv')
sw_hours_rem_occuring.to_csv('icu_sleep_breathing_sw_hours_rem_occuring.csv')


In [56]:
sw_hours.loc['stage_pred_ecg_nn'].head()

,data,sleep,wake,R,night_data,night_sleep,night_wake,day_data,day_sleep,day_wake,day_r,night_r,day_sleep_proportion,night_sleep_proportion,sleep_day_proportion,sleep_night_proportion,day_sleep_per24hours,day_wake_per24hours,night_sleep_per24hours,night_wake_per24hours,day_r_per24hours,night_r_per24hours,sleep_day_proportion_adjusted24hours,sleep_night_proportion_adjusted24hours
study_id,,,,,,,,,,,,,,,,,,,,,,,,
035,4601,1376,3225,0,2396,1078,1318,2205,298,1907,0,0,13.51,44.99,21.66,78.34,0.027029,0.172971,0.089983,0.110017,0.000000,0.000000,23.10,76.90
050,4350,1663,2687,22,2387,1320,1067,1963,343,1620,2,20,17.47,55.30,20.63,79.37,0.034947,0.165053,0.110599,0.089401,0.000204,0.001676,24.01,75.99
001,7506,1297,6209,144,3840,923,2917,3666,374,3292,55,89,10.20,24.04,28.84,71.16,0.020404,0.179596,0.048073,0.151927,0.003001,0.004635,29.80,70.20
015,13249,1293,11956,0,6577,973,5604,6672,320,6352,0,0,4.80,14.79,24.75,75.25,0.009592,0.190408,0.029588,0.170412,0.000000,0.000000,24.48,75.52
123,4130,903,3227,31,2106,835,1271,2024,68,1956,0,31,3.36,39.65,7.53,92.47,0.006719,0.193281,0.079297,0.120703,0.000000,0.002944,7.81,92.19


In [57]:
df_sleep_periods = pd.DataFrame(index=swimmer.index)
sleep_periods_lengths_all_subjects = []

In [58]:
swimmer

,,2020-01-01 00:00:00,2020-01-01 00:00:30,2020-01-01 00:01:00,2020-01-01 00:01:30,2020-01-01 00:02:00,2020-01-01 00:02:30,2020-01-01 00:03:00,2020-01-01 00:03:30,2020-01-01 00:04:00,2020-01-01 00:04:30,2020-01-01 00:05:00,2020-01-01 00:05:30,2020-01-01 00:06:00,2020-01-01 00:06:30,2020-01-01 00:07:00,2020-01-01 00:07:30,2020-01-01 00:08:00,2020-01-01 00:08:30,2020-01-01 00:09:00,2020-01-01 00:09:30,2020-01-01 00:10:00,2020-01-01 00:10:30,2020-01-01 00:11:00,2020-01-01 00:11:30,2020-01-01 00:12:00,2020-01-01 00:12:30,2020-01-01 00:13:00,2020-01-01 00:13:30,2020-01-01 00:14:00,2020-01-01 00:14:30,2020-01-01 00:15:00,2020-01-01 00:15:30,2020-01-01 00:16:00,2020-01-01 00:16:30,2020-01-01 00:17:00,2020-01-01 00:17:30,2020-01-01 00:18:00,2020-01-01 00:18:30,2020-01-01 00:19:00,2020-01-01 00:19:30,2020-01-01 00:20:00,2020-01-01 00:20:30,2020-01-01 00:21:00,2020-01-01 00:21:30,2020-01-01 00:22:00,2020-01-01 00:22:30,2020-01-01 00:23:00,2020-01-01 00:23:30,2020-01-01 00:24:00,2020-01-01 00:24:30,2020-01-01 00:25:00,2020-01-01 00:25:30,2020-01-01 00:26:00,2020-01-01 00:26:30,2020-01-01 00:27:00,2020-01-01 00:27:30,2020-01-01 00:28:00,2020-01-01 00:28:30,2020-01-01 00:29:00,2020-01-01 00:29:30,2020-01-01 00:30:00,2020-01-01 00:30:30,2020-01-01 00:31:00,2020-01-01 00:31:30,2020-01-01 00:32:00,2020-01-01 00:32:30,2020-01-01 00:33:00,2020-01-01 00:33:30,2020-01-01 00:34:00,2020-01-01 00:34:30,2020-01-01 00:35:00,2020-01-01 00:35:30,2020-01-01 00:36:00,2020-01-01 00:36:30,2020-01-01 00:37:00,2020-01-01 00:37:30,2020-01-01 00:38:00,2020-01-01 00:38:30,2020-01-01 00:39:00,2020-01-01 00:39:30,2020-01-01 00:40:00,2020-01-01 00:40:30,2020-01-01 00:41:00,2020-01-01 00:41:30,2020-01-01 00:42:00,2020-01-01 00:42:30,2020-01-01 00:43:00,2020-01-01 00:43:30,2020-01-01 00:44:00,2020-01-01 00:44:30,2020-01-01 00:45:00,2020-01-01 00:45:30,2020-01-01 00:46:00,2020-01-01 00:46:30,2020-01-01 00:47:00,2020-01-01 00:47:30,2020-01-01 00:48:00,2020-01-01 00:48:30,2020-01-01 00:49:00,2020-01-01 00:49:30,2020-01-01 00:50:00,2020-01-01 00:50:30,2020-01-01 00:51:00,2020-01-01 00:51:30,2020-01-01 00:52:00,2020-01-01 00:52:30,2020-01-01 00:53:00,2020-01-01 00:53:30,2020-01-01 00:54:00,2020-01-01 00:54:30,2020-01-01 00:55:00,2020-01-01 00:55:30,2020-01-01 00:56:00,2020-01-01 00:56:30,2020-01-01 00:57:00,2020-01-01 00:57:30,2020-01-01 00:58:00,2020-01-01 00:58:30,2020-01-01 00:59:00,2020-01-01 00:59:30,2020-01-01 01:00:00,2020-01-01 01:00:30,2020-01-01 01:01:00,2020-01-01 01:01:30,2020-01-01 01:02:00,2020-01-01 01:02:30,2020-01-01 01:03:00,2020-01-01 01:03:30,2020-01-01 01:04:00,2020-01-01 01:04:30,2020-01-01 01:05:00,2020-01-01 01:05:30,2020-01-01 01:06:00,2020-01-01 01:06:30,2020-01-01 01:07:00,2020-01-01 01:07:30,2020-01-01 01:08:00,2020-01-01 01:08:30,2020-01-01 01:09:00,2020-01-01 01:09:30,2020-01-01 01:10:00,2020-01-01 01:10:30,2020-01-01 01:11:00,2020-01-01 01:11:30,2020-01-01 01:12:00,2020-01-01 01:12:30,2020-01-01 01:13:00,2020-01-01 01:13:30,2020-01-01 01:14:00,2020-01-01 01:14:30,...,2020-01-08 22:45:00,2020-01-08 22:45:30,2020-01-08 22:46:00,2020-01-08 22:46:30,2020-01-08 22:47:00,2020-01-08 22:47:30,2020-01-08 22:48:00,2020-01-08 22:48:30,2020-01-08 22:49:00,2020-01-08 22:49:30,2020-01-08 22:50:00,2020-01-08 22:50:30,2020-01-08 22:51:00,2020-01-08 22:51:30,2020-01-08 22:52:00,2020-01-08 22:52:30,2020-01-08 22:53:00,2020-01-08 22:53:30,2020-01-08 22:54:00,2020-01-08 22:54:30,2020-01-08 22:55:00,2020-01-08 22:55:30,2020-01-08 22:56:00,2020-01-08 22:56:30,2020-01-08 22:57:00,2020-01-08 22:57:30,2020-01-08 22:58:00,2020-01-08 22:58:30,2020-01-08 22:59:00,2020-01-08 22:59:30,2020-01-08 23:00:00,2020-01-08 23:00:30,2020-01-08 23:01:00,2020-01-08 23:01:30,2020-01-08 23:02:00,2020-01-08 23:02:30,2020-01-08 23:03:00,2020-01-08 23:03:30,2020-01-08 23:04:00,2020-01-08 23:04:30,2020-01-08 23:05:00,2020-01-08 23:05:30,2020-01-08 23:06:00,2020-01-08 23:06:30,2020-01-08 23:07:00,2020-01-08 23:07:30,2020-01-08 23:08:00,2020-01-08 23:08:30,2020-01-08 23:09:00,2020-01-08 23:

In [59]:
df_sleep_periods['num_sleep_periods_24hr'] = np.nan # not tested yet. can be removed otherwise with a line further down.

for min_duration_sleep_period in [1, 5]:

    sleep_periods_lengths_all_subjects = {}
    for stage_col in ['stage_pred_amendsumeffort', 'stage_pred_amendsumeffort_agrelaxed',
           'stage_pred_ecg_nn', 'stage_pred_ecg_nn_agrelaxed']:
        sleep_periods_lengths_all_subjects[stage_col] = []

    for jloc, row in swimmer.iterrows():

        # remove no-signal parts:
        data_subject = row[row!=-1].copy()
        data_subject = np.isin(data_subject, [1, 2, 3, 4]).astype(int)

        hours_data = len(data_subject) * 2 / 60
        df_sleep_periods.loc[row.name, 'hours_data'] = hours_data

        sleep_periods = np.split(data_subject, np.where(data_subject == 0)[0]) # split by 0/wake
        sleep_periods = [x[1:] for x in sleep_periods if len(x) > min_duration_sleep_period *2] # 30 second epochs, so *2

        df_sleep_periods.loc[row.name, 'num_sleep_periods'] = len(sleep_periods)

        sleep_periods_lengths = [len(x)*30/60 for x in sleep_periods]
        sleep_periods_lengths_all_subjects[row.name[0]].extend(sleep_periods_lengths)

        if len(sleep_periods_lengths) == 0: continue
        df_sleep_periods.loc[row.name, 'sleep_periods_length_mean'] = np.mean(sleep_periods_lengths)
        df_sleep_periods.loc[row.name, 'sleep_periods_length_std'] = np.std(sleep_periods_lengths)
        df_sleep_periods.loc[row.name, 'sleep_periods_length_min'] = np.min(sleep_periods_lengths)
        df_sleep_periods.loc[row.name, 'sleep_periods_length_max'] = np.max(sleep_periods_lengths)
    print('_______________________________________________________________')
    print(f'With sleep period minimum {min_duration_sleep_period} minutes.')
    df_sleep_periods.to_csv(f'icu_sleep_breathing_sleep_periods_{min_duration_sleep_period}.csv')
    for stage_col in ['stage_pred_amendsumeffort', 'stage_pred_amendsumeffort_agrelaxed',
           'stage_pred_ecg_nn', 'stage_pred_ecg_nn_agrelaxed']:
        print(f"\n{stage_col}")

        df_sleep_periods.loc[stage_col, 'num_sleep_periods_24hr'] = (df_sleep_periods.loc[stage_col, 'num_sleep_periods']/df_sleep_periods.loc[stage_col, 'hours_data']*24).values
        print(f"Number of sleep periods Mean (SD) : {df_sleep_periods.loc[stage_col, 'num_sleep_periods'].mean().round(2)} ({df_sleep_periods.loc[stage_col, 'num_sleep_periods'].std().round(2)})")
        print(f"Number of sleep periods / 24 hour of data Mean (SD) : {(df_sleep_periods.loc[stage_col, 'num_sleep_periods']/df_sleep_periods.loc[stage_col, 'hours_data']*24).mean().round(2)} ({(df_sleep_periods.loc[stage_col, 'num_sleep_periods']/df_sleep_periods.loc[stage_col, 'hours_data']*24).std().round(2)})")

_______________________________________________________________
With sleep period minimum 1 minutes.

stage_pred_amendsumeffort
Number of sleep periods Mean (SD) : 34.79 (35.73)
Number of sleep periods / 24 hour of data Mean (SD) : 5.6 (3.95)

stage_pred_amendsumeffort_agrelaxed
Number of sleep periods Mean (SD) : 58.08 (65.63)
Number of sleep periods / 24 hour of data Mean (SD) : 11.8 (6.98)

stage_pred_ecg_nn
Number of sleep periods Mean (SD) : 82.32 (63.66)
Number of sleep periods / 24 hour of data Mean (SD) : 8.1 (4.71)

stage_pred_ecg_nn_agrelaxed
Number of sleep periods Mean (SD) : 38.26 (44.43)
Number of sleep periods / 24 hour of data Mean (SD) : 8.46 (6.39)
_______________________________________________________________
With sleep period minimum 5 minutes.

stage_pred_amendsumeffort
Number of sleep periods Mean (SD) : 17.82 (17.7)
Number of sleep periods / 24 hour of data Mean (SD) : 2.98 (2.0)

stage_pred_amendsumeffort_agrelaxed
Number of sleep periods Mean (SD) : 20.06 (22.

In [60]:
print('over all patients:')
for stage_col in sleep_periods_lengths_all_subjects.keys():
    print(f'\n{stage_col}')
    print(f'Mean: {np.mean(sleep_periods_lengths_all_subjects[stage_col]).round(1)} and STD {np.std(sleep_periods_lengths_all_subjects[stage_col]).round(1)}')
    print(f'Range: {np.min(sleep_periods_lengths_all_subjects[stage_col])} - {np.max(sleep_periods_lengths_all_subjects[stage_col])}')
    print(f'1/99% quantile range: {np.quantile(sleep_periods_lengths_all_subjects[stage_col], 0.01).round(1)} - {np.quantile(sleep_periods_lengths_all_subjects[stage_col], 0.99).round(1)}')

over all patients:

stage_pred_amendsumeffort
Mean: 78.9 and STD 209.4
Range: 5.0 - 3693.5
1/99% quantile range: 5.0 - 889.6

stage_pred_amendsumeffort_agrelaxed
Mean: 17.0 and STD 19.4
Range: 5.0 - 344.0
1/99% quantile range: 5.0 - 83.8

stage_pred_ecg_nn
Mean: 28.5 and STD 53.1
Range: 5.0 - 1465.5
1/99% quantile range: 5.0 - 223.4

stage_pred_ecg_nn_agrelaxed
Mean: 15.5 and STD 17.5
Range: 5.0 - 233.5
1/99% quantile range: 5.0 - 82.7


In [ ]:
# get number and average/sd of each sleep period.

https://www.atsjournals.org/doi/full/10.1164/ajrccm.163.2.9912128  
fifty-seven percent ± 18% of the total sleep time occurred during the daytime (6:00 a.m.–10:00 p.m.) and 43 ± 18% of the TST occurred during the nighttime hours (10:00 p.m.–6:00 a.m.).  
OUR DATA: we use 8am-8pm, 8pm-8am periods. 30% of sleep during day, 70% during night, if we use agreed sleep only. all hrv-model based sleep 36% vs. 63%.


The mean number of sleep periods per 24-h study period was 41 ± 28 (range 5–100). The mean length of each sleep bout was 15 ± 9 min (range 5.5–40 min). REM sleep, when it occurred, was equally distributed between day (50 ± 5%) and night (50 ± 5%). The overall arousal index (number of arousals per hour of sleep) was normal (mean 11.6 ± 5.0; range 4.6–21.2) (22).